<a href="https://colab.research.google.com/github/alamar17/Deep-Learning-Projects/blob/main/PyTorch_Wine_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
PROJECT: Multivariate Wine Quality Classification
ARCHITECTURE: Multi-Layer Perceptron (MLP) using PyTorch
GOAL: Predicting wine class based on 13 chemical constituents
"""


# Import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np
import random


# Data Loading
from google.colab import drive
drive.mount('/content/drive')
data_path = "/content/drive/My Drive/Deep Learning/wine.csv"
df = pd.read_csv(data_path)
print(df.head())



# Reproducibility Setup
# Fixing the seed ensures that the weight initialization and data shuffling
# are identical across runs, locking the accuracy (e.g., 91.67%) for consistency
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


x = df.drop('Wine', axis=1).values
y = df['Wine'].values


# Encode class labels (1, 2, 3) into 0-indexed values (0, 1, 2)
# Necessary because PyTorch's CrossEntropyLoss expects a 0 to N-1 range.

le = LabelEncoder()
y = le.fit_transform(y)

# Data Preprocessing
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("Unique labels in training set:", set(y_train))
print("Unique labels in testing set:", set(y_test))

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train = torch.FloatTensor(x_train)
x_test = torch.FloatTensor(x_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

class WineClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(WineClassifier, self).__init__()
    self.layer1 = nn.Linear(input_dim, hidden_dim)
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)
    self.layer3 = nn.Linear(hidden_dim, output_dim)

  def forward(self, x):
    x = torch.relu(self.layer1(x))
    x = torch.relu(self.layer2(x))
    x = self.layer3(x)
    return x


model = WineClassifier(input_dim=x_train.shape[1], hidden_dim=10, output_dim=len(set(y)))

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


num_epochs = 100
for epoch in range(num_epochs):
  outputs = model(x_train)


  loss = criterion(outputs, y_train)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if (epoch+1) % 10 == 0:
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

with torch.no_grad():
  outputs = model(x_test)
  _, predicted = torch.max(outputs, 1)
  accuracy = (predicted == y_test).sum().item() / len(y_test)
  print(f"Test Accuracy: {accuracy * 100:.2f}%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Wine  Alcohol  Malic.acid   Ash   Acl   Mg  Phenols  Flavanoids  \
0     1    14.23        1.71  2.43  15.6  127     2.80        3.06   
1     1    13.20        1.78  2.14  11.2  100     2.65        2.76   
2     1    13.16        2.36  2.67  18.6  101     2.80        3.24   
3     1    14.37        1.95  2.50  16.8  113     3.85        3.49   
4     1    13.24        2.59  2.87  21.0  118     2.80        2.69   

   Nonflavanoid.phenols  Proanth  Color.int   Hue    OD  Proline  
0                  0.28     2.29       5.64  1.04  3.92     1065  
1                  0.26     1.28       4.38  1.05  3.40     1050  
2                  0.30     2.81       5.68  1.03  3.17     1185  
3                  0.24     2.18       7.80  0.86  3.45     1480  
4                  0.39     1.82       4.32  1.04  2.93      735  
Unique labels in training set: {np.int64(0), np.

In [ ]:
"""
PROJECT: Multivariate Wine Quality Classification
ARCHITECTURE: Multi-Layer Perceptron (MLP) using PyTorch
GOAL: Predicting wine class based on 13 chemical constituents
"""


# Import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np
import random


# Data Loading
from google.colab import drive
drive.mount('/content/drive')
data_path = "/content/drive/My Drive/Deep Learning/wine.csv"
df = pd.read_csv(data_path)
print(df.head())

x = df.drop('Wine', axis=1).values
y = df['Wine'].values

# Reproducibility Setup
# Fixing the seed ensures that the weight initialization and data shuffling
# are identical across runs, locking the accuracy at 91.67% for consistency
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Encode class labels (1, 2, 3) into 0-indexed values (0, 1, 2)
# Necessary because PyTorch's CrossEntropyLoss expects a 0 to N-1 range.

le = LabelEncoder()
y = le.fit_transform(y)

# Data Preprocessing
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("Unique labels in training set:", set(y_train))
print("Unique labels in testing set:", set(y_test))

# Feature Scaling: Neural Networks converge much faster when inputs
# have a mean of 0 and a standard deviation of 1.

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Convert arrays to PyTorch Tensors (Floats for data, Longs for integer labels)
x_train = torch.FloatTensor(x_train)
x_test = torch.FloatTensor(x_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)


# Model Architecture
class WineClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(WineClassifier, self).__init__()
    self.layer1 = nn.Linear(input_dim, hidden_dim)
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)
    self.layer3 = nn.Linear(hidden_dim, output_dim)

  def forward(self, x):
    # ReLU acts as a biological threshold; only "fires" if signal is > 0
    x = torch.relu(self.layer1(x))
    x = torch.relu(self.layer2(x))
    # Final layer returns 'logits' (raw scores for each of the 3 wine classes)
    x = self.layer3(x)
    return x

# Initialize model (13 inputs -> 10 hidden neurons -> 3 outputs)
model = WineClassifier(input_dim=x_train.shape[1], hidden_dim=10, output_dim=len(set(y)))

# Define the "Grader" (Loss) and the "Coach" (Optimizer)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 100
for epoch in range(num_epochs):
  # Forward Pass: Model makes a guess
  outputs = model(x_train)
  loss = criterion(outputs, y_train)

  # Backwards Pass: 3-step optimization process
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if (epoch+1) % 10 == 0:
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
# Evaluation
# Disable gradient tracking to save memory during testing
with torch.no_grad():
  outputs = model(x_test)
  # Pick the class with the highest probability score
  _, predicted = torch.max(outputs, 1)
  accuracy = (predicted == y_test).sum().item() / len(y_test)
  print(f"Test Accuracy: {accuracy * 100:.2f}%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Wine  Alcohol  Malic.acid   Ash   Acl   Mg  Phenols  Flavanoids  \
0     1    14.23        1.71  2.43  15.6  127     2.80        3.06   
1     1    13.20        1.78  2.14  11.2  100     2.65        2.76   
2     1    13.16        2.36  2.67  18.6  101     2.80        3.24   
3     1    14.37        1.95  2.50  16.8  113     3.85        3.49   
4     1    13.24        2.59  2.87  21.0  118     2.80        2.69   

   Nonflavanoid.phenols  Proanth  Color.int   Hue    OD  Proline  
0                  0.28     2.29       5.64  1.04  3.92     1065  
1                  0.26     1.28       4.38  1.05  3.40     1050  
2                  0.30     2.81       5.68  1.03  3.17     1185  
3                  0.24     2.18       7.80  0.86  3.45     1480  
4                  0.39     1.82       4.32  1.04  2.93      735  
Unique labels in training set: {np.int64(0), np.